<h1><b> Librerías y configuraciones

In [1]:
# Librerías
import sys
import copy
import math
import torch
import torch.nn as nn
import numpy as np
import random
import importlib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import torchvision.transforms.functional as TF

from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import roc_auc_score, confusion_matrix, f1_score, recall_score, precision_score

# Modelos
sys.path.append(str(Path(".").resolve()))

import agregacion
importlib.reload(agregacion)
from agregacion import agregar_pesos

from modelo import crear_modelo

# Semilla para reproducibilidad
SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)

# Configuración del dispositivo
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")

Dispositivo: cuda


In [2]:
BASE_DIR = Path("..")

CENTROS = {
    "center_1": {
        "train": BASE_DIR / "preprocesamiento/output/center_1/preprocesamiento_train",
        "test" : BASE_DIR / "preprocesamiento/output/center_1/preprocesamiento_test",
    },
    "center_2": {
        "train": BASE_DIR / "preprocesamiento/output/center_2/preprocesamiento_train",
        "test" : BASE_DIR / "preprocesamiento/output/center_2/preprocesamiento_test",
    },
    "center_3": {
        "train": BASE_DIR / "preprocesamiento/output/center_3/preprocesamiento_train",
        "test" : BASE_DIR / "preprocesamiento/output/center_3/preprocesamiento_test",
    },
    "center_4": {
        "train": BASE_DIR / "preprocesamiento/output/center_4/preprocesamiento_train",
        "test" : BASE_DIR / "preprocesamiento/output/center_4/preprocesamiento_test",
    },
}

# Directorio de pesos
PESOS_DIR = Path("ditto_pesos")
PESOS_DIR.mkdir(exist_ok=True)

# Hiperparámetros
BATCH_SIZE   = 16
EPOCHS_LOCAL = 20     # épocas entrenamiento global por ronda
EPOCHS_DITTO = 15     # épocas entrenamiento local Ditto por ronda
RONDAS       = 30
LR           = 1e-4
PACIENCIA    = 10
LAMBDA       = 0.5    # fuerza del ancla al modelo global

# workers para DataLoader
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# Verificar rutas
print("\nVerificación de rutas:")
for nombre, rutas in CENTROS.items():
    train_ok = rutas["train"].exists()
    test_ok  = rutas["test"].exists()
    print(f"  {nombre} — train: {'OK' if train_ok else 'BAD'} | test: {'OK' if test_ok else 'BAD'}")


Verificación de rutas:
  center_1 — train: OK | test: OK
  center_2 — train: OK | test: OK
  center_3 — train: OK | test: OK
  center_4 — train: OK | test: OK


<h1><b> Dataset

In [3]:
# Dataset
class StrokeDataset(Dataset):
    def __init__(self, archivos, augment=False):
        self.archivos = archivos
        self.augment  = augment

    def __len__(self):
        return len(self.archivos)

    def __getitem__(self, idx):
        ruta   = self.archivos[idx]
        imagen = np.load(ruta).astype(np.float32)
        imagen = torch.from_numpy(imagen)

        if self.augment:
            if random.random() > 0.5:
                imagen = TF.hflip(imagen)
            if random.random() > 0.5:
                imagen = TF.vflip(imagen)
            if random.random() > 0.5:
                angle  = random.uniform(-15, 15)
                imagen = TF.rotate(imagen, angle)

        label = 1 if "STROKE" in ruta.stem else 0
        return imagen, torch.tensor(label, dtype=torch.float32)


# Carga y split por centro
def cargar_centro(rutas_centro, seed=SEED):
    archivos_train = sorted(rutas_centro["train"].glob("*.npy"))
    archivos_test  = sorted(rutas_centro["test"].glob("*.npy"))

    # Split 80/20 sobre archivos — antes de crear datasets
    val_size   = int(len(archivos_train) * 0.2)
    train_size = len(archivos_train) - val_size

    indices = list(range(len(archivos_train)))
    rng     = random.Random(seed)
    rng.shuffle(indices)

    idx_train = indices[:train_size]
    idx_val   = indices[train_size:]

    arch_train = [archivos_train[i] for i in idx_train]
    arch_val   = [archivos_train[i] for i in idx_val]

    # Calcular pos_weight
    n_control  = sum(1 for f in arch_train if "CONTROL" in f.stem)
    n_stroke   = sum(1 for f in arch_train if "STROKE"  in f.stem)
    pos_weight = torch.tensor([n_stroke / n_control], dtype=torch.float32).to(DEVICE)

    dataset_train = StrokeDataset(arch_train, augment=True)   # ← augment en train
    dataset_val   = StrokeDataset(arch_val,   augment=False)  # ← val limpio
    dataset_test  = StrokeDataset(archivos_test, augment=False)

    g = torch.Generator()
    g.manual_seed(seed)

    loader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True,
                              worker_init_fn=seed_worker, generator=g)
    loader_val   = DataLoader(dataset_val,  batch_size=BATCH_SIZE, shuffle=False)
    loader_test  = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

    return {
        "loader_train" : loader_train,
        "loader_val"   : loader_val,
        "loader_test"  : loader_test,
        "n_train"      : train_size,
        "n_val"        : val_size,
        "n_test"       : len(archivos_test),
        "pos_weight"   : pos_weight,
    }


# Cargar todos los centros
datos_centros = {}
for nombre, rutas in CENTROS.items():
    datos_centros[nombre] = cargar_centro(rutas)

# Verificación
print(f"{'Centro':<12} {'Train':>8} {'Val':>8} {'Test':>8} {'pos_weight':>12}")
print("─" * 46)
for nombre, datos in datos_centros.items():
    print(f"{nombre:<12} {datos['n_train']:>8} {datos['n_val']:>8} {datos['n_test']:>8} {datos['pos_weight'].item():>12.4f}")

Centro          Train      Val     Test   pos_weight
──────────────────────────────────────────────
center_1         2432      608      760       0.9853
center_2          768      192      240       1.4000
center_3          544      135      171       1.7200
center_4         1024      256      320       1.1787


<h1><b>Modelos y loss por centro

In [4]:
# Punto de partida
set_seed(SEED)
modelo_global = crear_modelo().to(DEVICE)

In [5]:
# Modelos locales ditto
modelos_locales = {
    nombre: crear_modelo().to(DEVICE)
    for nombre in CENTROS.keys()
}

# Inicializamos los modelos locales con los pesos del global
for nombre, modelo_local in modelos_locales.items():
    modelo_local.load_state_dict(copy.deepcopy(modelo_global.state_dict()))

# Rutas de guardado
MODELO_GLOBAL_PATH = PESOS_DIR / "modelo_global_ditto.pth"
MODELOS_LOCALES_PATHS = {
    nombre: PESOS_DIR / f"{nombre}_local_ditto.pth"
    for nombre in CENTROS.keys()
}

In [6]:
# Loss por centro
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        bce   = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        pt    = torch.where(targets == 1, probs, 1 - probs)
        loss  = ((1 - pt) ** self.gamma) * bce
        return loss.mean()


LOSS_POR_CENTRO = {
    "center_1": nn.BCEWithLogitsLoss(),
    "center_2": nn.BCEWithLogitsLoss(),
    "center_3": nn.BCEWithLogitsLoss(pos_weight=datos_centros["center_3"]["pos_weight"]),
    "center_4": nn.BCEWithLogitsLoss(pos_weight=datos_centros["center_4"]["pos_weight"]),
}

# Verificación
total_params = sum(p.numel() for p in modelo_global.parameters())
print(f"Modelo global     : {total_params:,} parámetros")
print(f"Modelos locales   : {len(modelos_locales)} × {total_params:,} parámetros")
print(f"\nLoss por centro:")
for nombre, loss in LOSS_POR_CENTRO.items():
    pw = datos_centros[nombre]["pos_weight"].item()
    print(f"  {nombre} - {loss.__class__.__name__} | pos_weight: {pw:.4f}")

Modelo global     : 4,008,541 parámetros
Modelos locales   : 4 × 4,008,541 parámetros

Loss por centro:
  center_1 - BCEWithLogitsLoss | pos_weight: 0.9853
  center_2 - BCEWithLogitsLoss | pos_weight: 1.4000
  center_3 - BCEWithLogitsLoss | pos_weight: 1.7200
  center_4 - BCEWithLogitsLoss | pos_weight: 1.1787


<h1><b> Funciones de entrenamiento Ditto

In [7]:
def entrenar_global(nombre, modelo_global, loader_train, loader_val, epochs):

    modelo_local = copy.deepcopy(modelo_global)
    criterio     = LOSS_POR_CENTRO[nombre]
    optimizer    = torch.optim.AdamW(modelo_local.parameters(), lr=LR, weight_decay=1e-4)
    scheduler    = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    mejor_val_auc  = 0.0
    mejores_pesos  = None
    mejor_val_loss = float("inf")
    mejor_val_acc  = 0.0

    for epoca in range(epochs):
        modelo_local.train()
        train_loss, train_correctos = 0.0, 0

        for imagenes, labels in loader_train:
            imagenes = imagenes.to(DEVICE)
            labels   = labels.to(DEVICE).unsqueeze(1)
            optimizer.zero_grad()
            salida = modelo_local(imagenes)
            loss   = criterio(salida, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(modelo_local.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss      += loss.item()
            preds            = (torch.sigmoid(salida) >= 0.5).float()
            train_correctos += (preds == labels).sum().item()

        modelo_local.eval()
        val_loss, val_correctos = 0.0, 0
        todas_probs  = []
        todos_labels = []

        with torch.no_grad():
            for imagenes, labels in loader_val:
                imagenes = imagenes.to(DEVICE)
                labels   = labels.to(DEVICE).unsqueeze(1)
                salida   = modelo_local(imagenes)
                loss     = criterio(salida, labels)
                probs    = torch.sigmoid(salida).squeeze(1).cpu().numpy()
                val_loss      += loss.item()
                preds          = (torch.sigmoid(salida) >= 0.5).float()
                val_correctos += (preds == labels).sum().item()
                todas_probs.extend(probs)
                todos_labels.extend(labels.squeeze(1).cpu().numpy())

        train_loss /= len(loader_train)
        val_loss   /= len(loader_val)
        train_acc   = train_correctos / len(loader_train.dataset)
        val_acc     = val_correctos   / len(loader_val.dataset)
        val_auc     = roc_auc_score(todos_labels, todas_probs)

        scheduler.step()

        print(f"      [global/{nombre}] época {epoca+1:02d}/{epochs} — "
              f"loss: {train_loss:.4f} acc: {train_acc:.4f} | "
              f"val_loss: {val_loss:.4f} acc: {val_acc:.4f} | auc: {val_auc:.4f}")

        if val_auc > mejor_val_auc:
            mejor_val_auc  = val_auc
            mejor_val_loss = val_loss
            mejor_val_acc  = val_acc
            mejores_pesos  = copy.deepcopy(modelo_local.state_dict())
            print(f"        → mejor época global: {epoca+1} | auc: {mejor_val_auc:.4f}")

    return mejores_pesos, mejor_val_loss, mejor_val_acc, mejor_val_auc

In [8]:
def entrenar_ditto(nombre, modelo_local, modelo_global, loader_train, loader_val, epochs, lam=LAMBDA):
    """
    Personaliza el modelo local de cada centro.
    El ancla al global evita que se aleje demasiado.
    Retorna val_loss, val_acc, val_auc del mejor modelo local.
    """
    criterio  = LOSS_POR_CENTRO[nombre]
    optimizer = torch.optim.AdamW(modelo_local.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # Pesos del global como referencia fija
    pesos_global_ref = {k: v.clone().detach() for k, v in modelo_global.state_dict().items()}

    mejor_val_auc  = 0.0
    mejor_estado   = None
    mejor_val_loss = float("inf")
    mejor_val_acc  = 0.0

    for epoca in range(epochs):
        modelo_local.train()
        train_loss, train_correctos = 0.0, 0

        for imagenes, labels in loader_train:
            imagenes = imagenes.to(DEVICE)
            labels   = labels.to(DEVICE).unsqueeze(1)
            optimizer.zero_grad()
            salida      = modelo_local(imagenes)
            loss_normal = criterio(salida, labels)

            # Ancla al modelo global
            ancla = 0.0
            for k, param in modelo_local.named_parameters():
                if k in pesos_global_ref:
                    ancla += ((param - pesos_global_ref[k]) ** 2).sum()
            loss_total = loss_normal + (lam / 2) * ancla

            loss_total.backward()
            torch.nn.utils.clip_grad_norm_(modelo_local.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss      += loss_normal.item()
            preds            = (torch.sigmoid(salida) >= 0.5).float()
            train_correctos += (preds == labels).sum().item()

        modelo_local.eval()
        val_loss, val_correctos = 0.0, 0
        todas_probs  = []
        todos_labels = []

        with torch.no_grad():
            for imagenes, labels in loader_val:
                imagenes = imagenes.to(DEVICE)
                labels   = labels.to(DEVICE).unsqueeze(1)
                salida   = modelo_local(imagenes)
                loss     = criterio(salida, labels)
                probs    = torch.sigmoid(salida).squeeze(1).cpu().numpy()
                val_loss      += loss.item()
                preds          = (torch.sigmoid(salida) >= 0.5).float()
                val_correctos += (preds == labels).sum().item()
                todas_probs.extend(probs)
                todos_labels.extend(labels.squeeze(1).cpu().numpy())

        train_loss /= len(loader_train)
        val_loss   /= len(loader_val)
        train_acc   = train_correctos / len(loader_train.dataset)
        val_acc     = val_correctos   / len(loader_val.dataset)
        val_auc     = roc_auc_score(todos_labels, todas_probs)

        scheduler.step()

        print(f"      [ditto/{nombre}] época {epoca+1:02d}/{epochs} — "
              f"loss: {train_loss:.4f} acc: {train_acc:.4f} | "
              f"val_loss: {val_loss:.4f} acc: {val_acc:.4f} | auc: {val_auc:.4f}")

        if val_auc > mejor_val_auc:
            mejor_val_auc  = val_auc
            mejor_val_loss = val_loss
            mejor_val_acc  = val_acc
            mejor_estado   = copy.deepcopy(modelo_local.state_dict())
            print(f"        → mejor época local: {epoca+1} | auc: {mejor_val_auc:.4f}")

    # Cargar mejor estado en el modelo local
    if mejor_estado is not None:
        modelo_local.load_state_dict(mejor_estado)

    return mejor_val_loss, mejor_val_acc, mejor_val_auc

<h1><b>Entrenamiento

In [9]:
# Paso 1: cada centro entrena copia del global - FedAvg
# Paso 2: cada centro personaliza su modelo local - Ditto

mejor_val_global    = 0.0
mejores_val_locales = {nombre: 0.0 for nombre in CENTROS.keys()}
rondas_sin_mejora   = 0
total_muestras      = sum(d["n_train"] for d in datos_centros.values())

historial = {"ronda": [], "val_auc_global": [], "val_auc_locales": []}

for ronda in range(1, RONDAS + 1):

    print(f"\n{'='*60}")
    print(f"  RONDA {ronda}/{RONDAS}")
    print(f"{'='*60}")

    set_seed(SEED + ronda)

    # Reiniciar generator
    g = torch.Generator()
    g.manual_seed(SEED + ronda)
    for nombre, datos in datos_centros.items():
        datos["loader_train"] = DataLoader(
            datos["loader_train"].dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            worker_init_fn=seed_worker,
            generator=g
        )

    # Entrenamiento global
    print(f"\n  ── Entrenamiento global ──")
    pesos_centros    = []
    val_aucs_global  = []
    muestras_centros = []

    for nombre, datos in datos_centros.items():
        print(f"\n  Centro: {nombre}")
        set_seed(SEED + ronda)
        pesos, val_loss, val_acc, val_auc = entrenar_global(
            nombre,
            modelo_global,
            datos["loader_train"],
            datos["loader_val"],
            EPOCHS_LOCAL
        )
        pesos_centros.append(pesos)
        val_aucs_global.append(val_auc)
        muestras_centros.append(datos["n_train"])

    # FedAvg — actualizar modelo global
    pesos_globales = agregar_pesos(pesos_centros, muestras_centros)
    modelo_global.load_state_dict(pesos_globales)

    # Val global ponderada por AUC
    val_global = sum(
        va * (n / total_muestras)
        for va, n in zip(val_aucs_global, muestras_centros)
    )

    # Guardar mejor modelo global + early stopping
    if val_global > mejor_val_global:
        mejor_val_global  = val_global
        rondas_sin_mejora = 0
        torch.save(modelo_global.state_dict(), MODELO_GLOBAL_PATH)
        print(f"\n  - Mejor modelo global guardado (val_auc: {mejor_val_global:.4f})")

        # ── Personalización Ditto — solo si el global mejoró ──
        print(f"\n  ── Personalización Ditto ──")
        val_aucs_locales = {}

        for nombre, datos in datos_centros.items():
            print(f"\n  Centro: {nombre}")
            set_seed(SEED + ronda)
            val_loss, val_acc, val_auc = entrenar_ditto(
                nombre,
                modelos_locales[nombre],
                modelo_global,
                datos["loader_train"],
                datos["loader_val"],
                EPOCHS_DITTO
            )
            val_aucs_locales[nombre] = val_auc

            if val_auc > mejores_val_locales[nombre]:
                mejores_val_locales[nombre] = val_auc
                torch.save(modelos_locales[nombre].state_dict(),
                           MODELOS_LOCALES_PATHS[nombre])
                print(f"    - Mejor modelo local guardado | auc: {val_auc:.4f}")

    else:
        rondas_sin_mejora += 1
        print(f"  Sin mejora global {rondas_sin_mejora}/{PACIENCIA}")
        val_aucs_locales = {nombre: mejores_val_locales[nombre] for nombre in CENTROS.keys()}

    # Resumen de ronda
    historial["ronda"].append(ronda)
    historial["val_auc_global"].append(val_global)
    historial["val_auc_locales"].append(val_aucs_locales)

    print(f"\n  {'─'*55}")
    print(f"  Ronda {ronda} — val_auc global: {val_global:.4f}")
    print(f"  Val AUC locales:")
    for nombre, auc in val_aucs_locales.items():
        print(f"    {nombre} → {auc:.4f}")

    # Early stopping
    if rondas_sin_mejora >= PACIENCIA:
        print(f"\n  Early stopping en ronda {ronda}")
        break

print(f"\n{'='*60}")
print(f"Entrenamiento Ditto finalizado")
print(f"Mejor val_auc global : {mejor_val_global:.4f}")
print(f"Mejores val_auc locales:")
for nombre, auc in mejores_val_locales.items():
    print(f"  {nombre} -> {auc:.4f}")


  RONDA 1/30

  ── Entrenamiento global ──

  Centro: center_1
      [global/center_1] época 01/20 — loss: 0.3158 acc: 0.8725 | val_loss: 0.1142 acc: 0.9589 | auc: 0.9959
        → mejor época global: 1 | auc: 0.9959
      [global/center_1] época 02/20 — loss: 0.1382 acc: 0.9548 | val_loss: 0.0556 acc: 0.9819 | auc: 0.9989
        → mejor época global: 2 | auc: 0.9989
      [global/center_1] época 03/20 — loss: 0.0912 acc: 0.9667 | val_loss: 0.0419 acc: 0.9885 | auc: 0.9989
        → mejor época global: 3 | auc: 0.9989
      [global/center_1] época 04/20 — loss: 0.0924 acc: 0.9716 | val_loss: 0.0403 acc: 0.9901 | auc: 0.9994
        → mejor época global: 4 | auc: 0.9994
      [global/center_1] época 05/20 — loss: 0.0556 acc: 0.9811 | val_loss: 0.0500 acc: 0.9836 | auc: 0.9992
      [global/center_1] época 06/20 — loss: 0.0436 acc: 0.9868 | val_loss: 0.0269 acc: 0.9934 | auc: 0.9995
        → mejor época global: 6 | auc: 0.9995
      [global/center_1] época 07/20 — loss: 0.0477 acc: 0.

<h1><b> Evaluación

In [10]:
def evaluar_modelo(nombre, modelo, loader_test):
    modelo.eval()
    todos_labels = []
    todos_probs  = []
    todos_preds  = []

    with torch.no_grad():
        for imagenes, labels in loader_test:
            imagenes = imagenes.to(DEVICE)
            salida   = modelo(imagenes)
            probs    = torch.sigmoid(salida).squeeze(1).cpu().numpy()
            preds    = (probs >= 0.5).astype(int)
            todos_labels.extend(labels.numpy().astype(int))
            todos_probs.extend(probs)
            todos_preds.extend(preds)

    return {
        "auc"          : roc_auc_score(todos_labels, todos_probs),
        "f1"           : f1_score(todos_labels, todos_preds),
        "sensibilidad" : recall_score(todos_labels, todos_preds),
        "especificidad": recall_score(todos_labels, todos_preds, pos_label=0),
        "precision"    : precision_score(todos_labels, todos_preds),
        "accuracy"     : sum(p == l for p, l in zip(todos_preds, todos_labels)) / len(todos_labels),
        "cm"           : confusion_matrix(todos_labels, todos_preds),
    }

# Cargar mejor modelo global
modelo_global.load_state_dict(torch.load(MODELO_GLOBAL_PATH, map_location=DEVICE))

# Cargar mejores modelos locales
for nombre in CENTROS.keys():
    modelos_locales[nombre].load_state_dict(
        torch.load(MODELOS_LOCALES_PATHS[nombre], map_location=DEVICE)
    )

# Evaluar
resultados_global = {}
resultados_local  = {}

for nombre, datos in datos_centros.items():
    resultados_global[nombre] = evaluar_modelo(nombre, modelo_global, datos["loader_test"])
    resultados_local[nombre]  = evaluar_modelo(nombre, modelos_locales[nombre], datos["loader_test"])

# Tabla comparativa
metricas = ["auc", "accuracy", "f1", "sensibilidad", "especificidad", "precision"]
nombres  = ["AUC-ROC", "Accuracy", "F1-Score", "Sensibilidad", "Especificidad", "Precisión"]

for nombre in CENTROS.keys():
    print(f"\n{'='*55}")
    print(f"  {nombre.upper()}")
    print(f"{'='*55}")
    print(f"  {'Métrica':<16} {'Global':>12} {'Local Ditto':>12}")
    print(f"  {'─'*42}")
    for metrica, nombre_m in zip(metricas, nombres):
        vg = resultados_global[nombre][metrica]
        vl = resultados_local[nombre][metrica]
        mejor = "←" if vl > vg else ""
        print(f"  {nombre_m:<16} {vg:>12.4f} {vl:>12.4f}  {mejor}")

# Resumen AUC
print(f"\n{'='*55}")
print(f"  RESUMEN AUC-ROC")
print(f"{'='*55}")
print(f"  {'Centro':<12} {'Global':>10} {'Local':>10} {'Mejor':>10}")
print(f"  {'─'*45}")
for nombre in CENTROS.keys():
    vg = resultados_global[nombre]["auc"]
    vl = resultados_local[nombre]["auc"]
    mejor = "Local" if vl > vg else "Global"
    print(f"  {nombre:<12} {vg:>10.4f} {vl:>10.4f} {mejor:>10}")


  CENTER_1
  Métrica                Global  Local Ditto
  ──────────────────────────────────────────
  AUC-ROC                0.9841       0.9848  ←
  Accuracy               0.9487       0.9447  
  F1-Score               0.9483       0.9432  
  Sensibilidad           0.9421       0.9184  
  Especificidad          0.9553       0.9711  ←
  Precisión              0.9547       0.9694  ←

  CENTER_2
  Métrica                Global  Local Ditto
  ──────────────────────────────────────────
  AUC-ROC                0.9502       0.9928  ←
  Accuracy               0.7958       0.9500  ←
  F1-Score               0.7879       0.9559  ←
  Sensibilidad           0.6500       0.9286  ←
  Especificidad          1.0000       0.9800  
  Precisión              1.0000       0.9848  

  CENTER_3
  Métrica                Global  Local Ditto
  ──────────────────────────────────────────
  AUC-ROC                0.3419       0.5245  ←
  Accuracy               0.3743       0.5205  ←
  F1-Score               0.

<h1><b> Entrenamiento con lambda distintos

In [9]:
LAMBDA_POR_CENTRO = {
    "center_1": 0.1,
    "center_2": 0.1,
    "center_3": 0.5,
    "center_4": 0.8,
}

In [10]:
MODELO_GLOBAL_PATH = PESOS_DIR / "modelo_global_ditto_v2.pth"

MODELOS_LOCALES_PATHS = {
    nombre: PESOS_DIR / f"{nombre}_local_ditto_{str(LAMBDA_POR_CENTRO[nombre]).replace('.', '')}.pth"
    for nombre in CENTROS.keys()
}

In [11]:
# Paso 1: cada centro entrena copia del global - FedAvg
# Paso 2: cada centro personaliza su modelo local - Ditto

mejor_val_global    = 0.0
mejores_val_locales = {nombre: 0.0 for nombre in CENTROS.keys()}
rondas_sin_mejora   = 0
total_muestras      = sum(d["n_train"] for d in datos_centros.values())

historial = {"ronda": [], "val_auc_global": [], "val_auc_locales": []}

for ronda in range(1, RONDAS + 1):

    print(f"\n{'='*60}")
    print(f"  RONDA {ronda}/{RONDAS}")
    print(f"{'='*60}")

    set_seed(SEED + ronda)

    # Reiniciar generator
    g = torch.Generator()
    g.manual_seed(SEED + ronda)
    for nombre, datos in datos_centros.items():
        datos["loader_train"] = DataLoader(
            datos["loader_train"].dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            worker_init_fn=seed_worker,
            generator=g
        )

    # Entrenamiento global
    print(f"\n  ── Entrenamiento global ──")
    pesos_centros    = []
    val_aucs_global  = []
    muestras_centros = []

    for nombre, datos in datos_centros.items():
        print(f"\n  Centro: {nombre}")
        set_seed(SEED + ronda)
        pesos, val_loss, val_acc, val_auc = entrenar_global(
            nombre,
            modelo_global,
            datos["loader_train"],
            datos["loader_val"],
            EPOCHS_LOCAL
        )
        pesos_centros.append(pesos)
        val_aucs_global.append(val_auc)
        muestras_centros.append(datos["n_train"])

    # FedAvg — actualizar modelo global
    pesos_globales = agregar_pesos(pesos_centros, muestras_centros)
    modelo_global.load_state_dict(pesos_globales)

    # Val global ponderada por AUC
    val_global = sum(
        va * (n / total_muestras)
        for va, n in zip(val_aucs_global, muestras_centros)
    )

    # Guardar mejor modelo global + early stopping
    if val_global > mejor_val_global:
        mejor_val_global  = val_global
        rondas_sin_mejora = 0
        torch.save(modelo_global.state_dict(), MODELO_GLOBAL_PATH)
        print(f"\n  - Mejor modelo global guardado (val_auc: {mejor_val_global:.4f})")

        # ── Personalización Ditto — solo si el global mejoró ──
        print(f"\n  ── Personalización Ditto ──")
        val_aucs_locales = {}

        for nombre, datos in datos_centros.items():
            print(f"\n  Centro: {nombre}")
            set_seed(SEED + ronda)
            val_loss, val_acc, val_auc = entrenar_ditto(
                nombre,
                modelos_locales[nombre],
                modelo_global,
                datos["loader_train"],
                datos["loader_val"],
                EPOCHS_DITTO,
                lam=LAMBDA_POR_CENTRO[nombre]
            )
            val_aucs_locales[nombre] = val_auc

            if val_auc > mejores_val_locales[nombre]:
                mejores_val_locales[nombre] = val_auc
                torch.save(modelos_locales[nombre].state_dict(),
                           MODELOS_LOCALES_PATHS[nombre])
                print(f"    - Mejor modelo local guardado | auc: {val_auc:.4f}")

    else:
        rondas_sin_mejora += 1
        print(f"  Sin mejora global {rondas_sin_mejora}/{PACIENCIA}")
        val_aucs_locales = {nombre: mejores_val_locales[nombre] for nombre in CENTROS.keys()}

    # Resumen de ronda
    historial["ronda"].append(ronda)
    historial["val_auc_global"].append(val_global)
    historial["val_auc_locales"].append(val_aucs_locales)

    print(f"\n  {'─'*55}")
    print(f"  Ronda {ronda} — val_auc global: {val_global:.4f}")
    print(f"  Val AUC locales:")
    for nombre, auc in val_aucs_locales.items():
        print(f"    {nombre} → {auc:.4f}")

    # Early stopping
    if rondas_sin_mejora >= PACIENCIA:
        print(f"\n  Early stopping en ronda {ronda}")
        break

print(f"\n{'='*60}")
print(f"Entrenamiento Ditto finalizado")
print(f"Mejor val_auc global : {mejor_val_global:.4f}")
print(f"Mejores val_auc locales:")
for nombre, auc in mejores_val_locales.items():
    print(f"  {nombre} -> {auc:.4f}")


  RONDA 1/30

  ── Entrenamiento global ──

  Centro: center_1
      [global/center_1] época 01/20 — loss: 0.2237 acc: 0.9054 | val_loss: 0.0802 acc: 0.9720 | auc: 0.9959
        → mejor época global: 1 | auc: 0.9959
      [global/center_1] época 02/20 — loss: 0.0673 acc: 0.9762 | val_loss: 0.0662 acc: 0.9819 | auc: 0.9965
        → mejor época global: 2 | auc: 0.9965
      [global/center_1] época 03/20 — loss: 0.0192 acc: 0.9938 | val_loss: 0.0851 acc: 0.9720 | auc: 0.9969
        → mejor época global: 3 | auc: 0.9969
      [global/center_1] época 04/20 — loss: 0.0238 acc: 0.9926 | val_loss: 0.0563 acc: 0.9786 | auc: 0.9990
        → mejor época global: 4 | auc: 0.9990
      [global/center_1] época 05/20 — loss: 0.0202 acc: 0.9926 | val_loss: 0.1468 acc: 0.9605 | auc: 0.9982
      [global/center_1] época 06/20 — loss: 0.0186 acc: 0.9938 | val_loss: 0.0493 acc: 0.9819 | auc: 0.9989
      [global/center_1] época 07/20 — loss: 0.0104 acc: 0.9955 | val_loss: 0.0179 acc: 0.9951 | auc: 0.9

In [12]:
modelo_global.load_state_dict(torch.load(MODELO_GLOBAL_PATH, map_location=DEVICE))

for nombre in CENTROS.keys():
    modelos_locales[nombre].load_state_dict(
        torch.load(MODELOS_LOCALES_PATHS[nombre], map_location=DEVICE)
    )

In [13]:
def evaluar_modelo(nombre, modelo, loader_test):
    modelo.eval()
    todos_labels = []
    todos_probs  = []
    todos_preds  = []

    with torch.no_grad():
        for imagenes, labels in loader_test:
            imagenes = imagenes.to(DEVICE)
            salida   = modelo(imagenes)
            probs    = torch.sigmoid(salida).squeeze(1).cpu().numpy()
            preds    = (probs >= 0.5).astype(int)
            todos_labels.extend(labels.numpy().astype(int))
            todos_probs.extend(probs)
            todos_preds.extend(preds)

    return {
        "auc"          : roc_auc_score(todos_labels, todos_probs),
        "f1"           : f1_score(todos_labels, todos_preds),
        "sensibilidad" : recall_score(todos_labels, todos_preds),
        "especificidad": recall_score(todos_labels, todos_preds, pos_label=0),
        "precision"    : precision_score(todos_labels, todos_preds),
        "accuracy"     : sum(p == l for p, l in zip(todos_preds, todos_labels)) / len(todos_labels),
        "cm"           : confusion_matrix(todos_labels, todos_preds),
    }

# Cargar mejor modelo global
modelo_global.load_state_dict(torch.load(MODELO_GLOBAL_PATH, map_location=DEVICE))

# Cargar mejores modelos locales
for nombre in CENTROS.keys():
    modelos_locales[nombre].load_state_dict(
        torch.load(MODELOS_LOCALES_PATHS[nombre], map_location=DEVICE)
    )

# Evaluar
resultados_global = {}
resultados_local  = {}

for nombre, datos in datos_centros.items():
    resultados_global[nombre] = evaluar_modelo(nombre, modelo_global, datos["loader_test"])
    resultados_local[nombre]  = evaluar_modelo(nombre, modelos_locales[nombre], datos["loader_test"])

# Tabla comparativa
metricas = ["auc", "accuracy", "f1", "sensibilidad", "especificidad", "precision"]
nombres  = ["AUC-ROC", "Accuracy", "F1-Score", "Sensibilidad", "Especificidad", "Precisión"]

for nombre in CENTROS.keys():
    print(f"\n{'='*55}")
    print(f"  {nombre.upper()}")
    print(f"{'='*55}")
    print(f"  {'Métrica':<16} {'Global':>12} {'Local Ditto':>12}")
    print(f"  {'─'*42}")
    for metrica, nombre_m in zip(metricas, nombres):
        vg = resultados_global[nombre][metrica]
        vl = resultados_local[nombre][metrica]
        mejor = "←" if vl > vg else ""
        print(f"  {nombre_m:<16} {vg:>12.4f} {vl:>12.4f}  {mejor}")

# Resumen AUC
print(f"\n{'='*55}")
print(f"  RESUMEN AUC-ROC")
print(f"{'='*55}")
print(f"  {'Centro':<12} {'Global':>10} {'Local':>10} {'Mejor':>10}")
print(f"  {'─'*45}")
for nombre in CENTROS.keys():
    vg = resultados_global[nombre]["auc"]
    vl = resultados_local[nombre]["auc"]
    mejor = "Local" if vl > vg else "Global"
    print(f"  {nombre:<12} {vg:>10.4f} {vl:>10.4f} {mejor:>10}")


  CENTER_1
  Métrica                Global  Local Ditto
  ──────────────────────────────────────────
  AUC-ROC                0.9661       0.9658  
  Accuracy               0.9171       0.9158  
  F1-Score               0.9168       0.9101  
  Sensibilidad           0.9132       0.8526  
  Especificidad          0.9211       0.9789  ←
  Precisión              0.9204       0.9759  ←

  CENTER_2
  Métrica                Global  Local Ditto
  ──────────────────────────────────────────
  AUC-ROC                0.8438       0.9790  ←
  Accuracy               0.7167       0.9500  ←
  F1-Score               0.7748       0.9552  ←
  Sensibilidad           0.8357       0.9143  ←
  Especificidad          0.5500       1.0000  ←
  Precisión              0.7222       1.0000  ←

  CENTER_3
  Métrica                Global  Local Ditto
  ──────────────────────────────────────────
  AUC-ROC                0.1673       0.7634  ←
  Accuracy               0.3099       0.7251  ←
  F1-Score               0

# **Inferencias center_5**

In [31]:
# Rutas
RUTA_CENTER_5 = Path("../preprocesamiento/output/center_5/preprocesamiento_test")

# Elección del modelo a evaluar
#RUTA_MODELO   = Path("ditto_pesos/center_4_local_ditto_08.pth")
RUTA_MODELO   = Path("ditto_pesos/center_4_local_ditto.pth")

NOMBRE_CSV    = "prob_ditto_c4_center_5.csv"
OUTPUT_DIR    = Path("probabilidades_center_5_ditto")
OUTPUT_DIR.mkdir(exist_ok=True)

In [32]:
# Sin etiquetas, solo probabilidades
class StrokeDatasetInferencia(Dataset):
    def __init__(self, archivos):
        self.archivos = archivos

    def __len__(self):
        return len(self.archivos)

    def __getitem__(self, idx):
        ruta   = self.archivos[idx]
        imagen = np.load(ruta).astype(np.float32)
        imagen = torch.from_numpy(imagen)
        return imagen, ruta.stem    

# Cargar datos
archivos_test = sorted(RUTA_CENTER_5.glob("*.npy"))
print(f"Archivos encontrados: {len(archivos_test)}")

dataset_test  = StrokeDatasetInferencia(archivos_test)
loader_test   = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

# Cargar modelo
modelo_inferencia = crear_modelo().to(DEVICE)
modelo_inferencia.load_state_dict(torch.load(RUTA_MODELO, map_location=DEVICE))
modelo_inferencia.eval()
print(f"Modelo cargado: {RUTA_MODELO.name}")

Archivos encontrados: 257
Modelo cargado: center_4_local_ditto.pth


In [33]:
# Inferencia
nombres_archivos = []
probabilidades   = []
predicciones     = []

with torch.no_grad():
    for imagenes, nombres in loader_test:
        imagenes = imagenes.to(DEVICE)
        salida   = modelo_inferencia(imagenes)
        probs    = torch.sigmoid(salida).squeeze(1).cpu().numpy()
        preds    = (probs >= 0.5).astype(int)

        nombres_archivos.extend(nombres)
        probabilidades.extend(probs.tolist())
        predicciones.extend(preds.tolist())

In [34]:
# Guardar resultados csv
df = pd.DataFrame({
    "archivo"      : nombres_archivos,
    "probabilidad" : probabilidades,
})

ruta_csv = OUTPUT_DIR / NOMBRE_CSV
df.to_csv(ruta_csv, index=False)

print(f"\nCSV guardado: {ruta_csv}")
print(f"Total muestras: {len(df)}")
print(f"\nDistribución predicciones:")
#print(df["clase"].value_counts())
print(f"\nPrimeras filas:")
print(df.head())


CSV guardado: probabilidades_center_5_ditto\prob_ditto_c4_center_5.csv
Total muestras: 257

Distribución predicciones:

Primeras filas:
       archivo  probabilidad
0  P0300_S0007  1.154062e-02
1  P0300_S0008  9.900782e-01
2  P0300_S0009  5.968570e-01
3  P0300_S0010  6.523227e-09
4  P0300_S0011  2.430261e-06
